In [1]:
"""
- Rekursives Laden aller *_E1.mat-Dateien
- Bandpassfilter 20–450 Hz
- Fenster: 400 Samples, Schrittweite 50 Samples
- Leave-One-Subject-Out (LOSO): eine Person bleibt komplett im Test
- Eine weitere Trainingsperson dient pro Fold als Validierung
- CNN mit Conv1D, BatchNorm, Pooling und Dropout
- GPU über PyTorch: Unter ROCm wird ebenfalls torch.device("cuda") verwendet
"""

from pathlib import Path
import re
import copy
import warnings

import numpy as np
import scipy.io as sio
from scipy.signal import butter, sosfiltfilt

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd


# ---------------------------------------------------------------------
# Einstellungen
# ---------------------------------------------------------------------

DATA_DIR = Path("./Data")      # z. B. Path("/home/walid/.../Data")
FILE_PATTERN = "*_E1.mat"

FS = 2000
LOWCUT = 20
HIGHCUT = 450

WINDOW_SIZE = 400
STEP_SIZE = 50

BATCH_SIZE = 64
EPOCHS = 60
EARLY_STOPPING_PATIENCE = 7
LR = 1e-3
WEIGHT_DECAY = 1e-4
RANDOM_SEED = 42

# Bandpassfilter wird auf das gesamte EMG-Signal jeder Datei angewendet
APPLY_BANDPASS = True

OUTPUT_CSV = Path("pytorch_loso_results.csv")

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# GPU-Beschleunigung
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    try:
        torch.cuda.manual_seed_all(RANDOM_SEED)
    except Exception:
        pass
else:
    DEVICE = torch.device("cpu")

print(f"PyTorch-Version: {torch.__version__}")
print(f"Gerät: {DEVICE}")
if DEVICE.type == "cuda":
    try:
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    except Exception:
        print("GPU erkannt (Gerätename konnte nicht gelesen werden)")
else:
    print("WARNUNG: Keine GPU erkannt; Training läuft auf der CPU")

# Daten laden und Fenster erzeugen

def get_subject_id(file_path: Path) -> int:
    match = re.search(r"S(\d+)", file_path.name, flags=re.IGNORECASE)
    if not match:
        raise ValueError(
            f"Personennummer konnte nicht aus dem Dateinamen gelesen werden: "
            f"{file_path.name}"
        )
    return int(match.group(1)) # Kann ich stark vereinfachen noch


def load_emg_and_labels(file_path: Path):
    data = sio.loadmat(file_path)

    if "emg" not in data:
        raise KeyError(f"'emg' fehlt in {file_path}")

    emg = np.asarray(data["emg"], dtype=np.float32)
    if emg.ndim != 2:
        raise ValueError(f"Unerwartete EMG-Form {emg.shape} in {file_path}")

    label_key = "restimulus" if "restimulus" in data else "stimulus"
    if label_key not in data:
        raise KeyError(f"Weder 'restimulus' noch 'stimulus' in {file_path}")

    labels = np.asarray(data[label_key]).reshape(-1)
    labels = np.rint(labels).astype(np.int64)
    n = min(len(emg), len(labels))
    emg = emg[:n]
    labels = labels[:n]

    if APPLY_BANDPASS:
        nyquist = FS / 2
        if not (0 < LOWCUT < HIGHCUT < nyquist):
            raise ValueError("Bandpass-Grenzen müssen zwischen 0 und Nyquist liegen")
        sos = butter(
            4, [LOWCUT, HIGHCUT], btype="bandpass", fs=FS, output="sos"
        )
        if len(emg) > 30:
            emg = sosfiltfilt(sos, emg, axis=0).astype(np.float32)

    return emg, labels


def make_windows(emg, labels, window_size=WINDOW_SIZE, step_size=STEP_SIZE):
    windows = []
    window_labels = []

    if len(emg) < window_size:
        return (
            np.empty((0, window_size, emg.shape[1]), dtype=np.float32),
            np.empty((0,), dtype=np.int64),
        )

    for start in range(0, len(emg) - window_size + 1, step_size):
        end = start + window_size
        y_window = labels[start:end]

        # Häufigstes Sample-Label des Fensters als Zielklasse
        values, counts = np.unique(y_window, return_counts=True)
        label = int(values[np.argmax(counts)])

        # Klasse 0 wird entfernt
        if label == 0:
            continue

        windows.append(emg[start:end])
        window_labels.append(label)

    if not windows:
        return (
            np.empty((0, window_size, emg.shape[1]), dtype=np.float32),
            np.empty((0,), dtype=np.int64),
        )

    return np.asarray(windows, dtype=np.float32), np.asarray(window_labels, dtype=np.int64)


def load_dataset(data_dir: Path):
    files = sorted(data_dir.rglob(FILE_PATTERN))

    if not files:
        raise FileNotFoundError(
            f"Keine Dateien mit Muster '{FILE_PATTERN}' unter '{data_dir.resolve()}' gefunden \n"
            "Prüfe DATA_DIR und ob die Dateien in Unterordnern liegen"
        )

    all_x = []
    all_y = []
    all_groups = []

    print(f"\nGefundene Dateien: {len(files)}")
    for i, file_path in enumerate(files, start=1):
        subject = get_subject_id(file_path)
        try:
            emg, labels = load_emg_and_labels(file_path)
            x_file, y_file = make_windows(emg, labels)

            if len(y_file) == 0:
                print(f"[{i}/{len(files)}] Übersprungen (keine aktiven Fenster): {file_path}")
                continue

            all_x.append(x_file)
            all_y.append(y_file)
            all_groups.append(np.full(len(y_file), subject, dtype=np.int64))

            print(
                f"[{i}/{len(files)}] {file_path.name}: "
                f"{len(y_file)} Fenster, {emg.shape[1]} EMG-Kanäle, Person {subject}"
            )
        except Exception as exc:
            warnings.warn(f"Datei übersprungen: {file_path} ({exc})")

    if not all_x:
        raise RuntimeError("Es konnten keine gültigen Fenster geladen werden")

    X = np.concatenate(all_x, axis=0).astype(np.float32)
    y = np.concatenate(all_y, axis=0).astype(np.int64)
    groups = np.concatenate(all_groups, axis=0).astype(np.int64)

    print("\nDatensatz:")
    print(f"X-Form: {X.shape}  (Fenster, Samples pro Fenster, Kanäle)")
    print(f"y-Form: {y.shape}")
    print(f"Personen: {np.unique(groups).tolist()}")
    print(f"Gestenklassen ohne Ruheklasse: {np.unique(y).tolist()}")

    return X, y, groups


# ---------------------------------------------------------------------
# CNN (Keras)
# ---------------------------------------------------------------------

class EMGCNN(nn.Module):
    def __init__(self, n_channels: int, n_classes: int):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(n_channels, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.15),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.15),

            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        # x-Form: (Batch, Kanäle, Samples)
        x = self.features(x)
        return self.classifier(x)



def normalize_by_training_channels(X_fit, X_other):

    mean = X_fit.mean(axis=(0, 1), keepdims=True)
    std = X_fit.std(axis=(0, 1), keepdims=True)
    std[std < 1e-6] = 1.0

    X_fit_norm = (X_fit - mean) / std
    X_other_norm = (X_other - mean) / std
    return X_fit_norm.astype(np.float32), X_other_norm.astype(np.float32), mean, std


def make_class_weights(y_fit_encoded, n_classes):
    counts = np.bincount(y_fit_encoded, minlength=n_classes).astype(np.float32)
    weights = np.zeros(n_classes, dtype=np.float32)
    present = counts > 0

    if np.any(present):
        # Inverse Häufigkeit für Klassen, die im Trainingsfold vorkommen.
        weights[present] = len(y_fit_encoded) / (
            np.count_nonzero(present) * counts[present]
        )

        # Gewichte auf Mittelwert 1 normieren.
        weights[present] /= weights[present].mean()

    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


# ---------------------------------------------------------------------
# Training und Validierung
# ---------------------------------------------------------------------

def make_loader(X, y, batch_size=BATCH_SIZE, shuffle=False):
    # Conv1D in PyTorch erwartet (Batch, Kanäle, Samples)
    X_tensor = torch.from_numpy(np.transpose(X, (0, 2, 1)).copy())
    y_tensor = torch.from_numpy(y.astype(np.int64))
    dataset = TensorDataset(X_tensor, y_tensor)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )



def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = torch.zeros((), device=DEVICE)
    total_correct = torch.zeros((), dtype=torch.long, device=DEVICE)
    total_items = 0

    context = torch.enable_grad() if training else torch.no_grad()

    with context:
        for batch_idx, (xb, yb) in enumerate(loader, start=1):
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            if training:
                optimizer.zero_grad(set_to_none=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            if training:
                loss.backward()
                optimizer.step()

            batch_n = yb.size(0)

            # Ergebnisse auf der GPU sammeln und erst am Ende auslesen
            total_loss += loss.detach() * batch_n
            total_correct += (
                (logits.detach().argmax(dim=1) == yb).sum()
            )
            total_items += batch_n

            # Alle 20 Batches bzw. am Ende Fortschritt anzeigen
            if batch_idx % 20 == 0 or batch_idx == len(loader):
                phase = "Training" if training else "Validierung"
                print(
                    f"  {phase}: Batch {batch_idx}/{len(loader)}",
                    flush=True
                )

    if total_items == 0:
        return float("inf"), 0.0

    return (
        total_loss.item() / total_items,
        total_correct.item() / total_items
    )



def predict(model, X, batch_size=BATCH_SIZE):
    model.eval()
    dummy_y = np.zeros(len(X), dtype=np.int64)
    loader = make_loader(X, dummy_y, batch_size=batch_size, shuffle=False)
    predictions = []

    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            logits = model(xb)
            predictions.append(logits.argmax(dim=1).cpu().numpy())

    if not predictions:
        return np.empty((0,), dtype=np.int64)

    return np.concatenate(predictions)


def train_one_fold(X_fit, y_fit, X_val, y_val, n_channels, n_classes, fold_no):
    train_loader = make_loader(X_fit, y_fit, shuffle=True)
    val_loader = make_loader(X_val, y_val, shuffle=False)

    model = EMGCNN(n_channels=n_channels, n_classes=n_classes).to(DEVICE)
    class_weights = make_class_weights(y_fit, n_classes)

# Klassengewichte nur für das Training
    criterion = nn.CrossEntropyLoss(weight=class_weights)

# Validierung ohne Klassengewichtung
    val_criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-6
    )

    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, optimizer=optimizer
        )
        val_loss, val_acc = run_epoch(model, val_loader, val_criterion)
        scheduler.step(val_loss)

        lr_now = optimizer.param_groups[0]["lr"]
        print(
            f"Fold {fold_no:02d} | Epoch {epoch:02d}/{EPOCHS} | "
            f"Train Loss {train_loss:.4f} Acc {train_acc:.4f} | "
            f"Val Loss {val_loss:.4f} Acc {val_acc:.4f} | LR {lr_now:.2e}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"Early Stopping nach Epoche {epoch}.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val_loss


# ---------------------------------------------------------------------
# Leave-One-Subject-Out
# ---------------------------------------------------------------------

def main():
    X, y_raw, groups = load_dataset(DATA_DIR)

    # LabelEncoder definiert nur die Zuordnung von Gesten-IDs zu
    # neuronalen Netzwerk-Ausgabeneuronen; Mittelwerte/Modelle werden
    # weiterhin ausschließlich mit Trainingspersonen gelernt
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y_raw)
    n_classes = len(label_encoder.classes_)
    n_channels = X.shape[2]

    logo = LeaveOneGroupOut()
    results = []
    all_true = []
    all_pred = []

    subjects = np.unique(groups)
    print(f"\nStarte LOSO: {len(subjects)} Personen, {n_classes} Klassen")
    print(f"Klassen-IDs (Original): {label_encoder.classes_.tolist()}")

    for fold_no, (trainval_idx, test_idx) in enumerate(
        logo.split(X, y, groups), start=1
    ):
        test_subject = int(groups[test_idx][0])
        trainval_subjects = np.unique(groups[trainval_idx])

        if len(trainval_subjects) < 2:
            raise RuntimeError(
                "Für einen unabhängigen Validierungssatz werden mindestens "
                "drei Personen insgesamt benötigt"
            )

        # Deterministische, pro Fold wechselnde Validierungsperson
        rng = np.random.default_rng(RANDOM_SEED + fold_no)
        val_subject = int(rng.choice(trainval_subjects))
        val_mask = np.isin(groups[trainval_idx], [val_subject])

        fit_idx = trainval_idx[~val_mask]
        val_idx = trainval_idx[val_mask]

        print("\n" + "=" * 78)
        print(f"Fold {fold_no}/{len(subjects)} | Testperson: {test_subject}")
        print(f"Validierungsperson: {val_subject}")
        print(f"Trainingspersonen: {np.unique(groups[fit_idx]).tolist()}")
        print(f"Train-Fenster: {len(fit_idx)} | Val-Fenster: {len(val_idx)} "
              f"| Test-Fenster: {len(test_idx)}")

        # Nur auf fit_idx normalisieren – weder Validierungs- noch Testdaten
        # beeinflussen die Skalierungsparameter.
        X_fit_norm, X_val_norm, mean, std = normalize_by_training_channels(
            X[fit_idx], X[val_idx]
        )
        X_test_norm = ((X[test_idx] - mean) / std).astype(np.float32)

        model, best_val_loss = train_one_fold(
            X_fit_norm,
            y[fit_idx],
            X_val_norm,
            y[val_idx],
            n_channels=n_channels,
            n_classes=n_classes,
            fold_no=fold_no,
        )

        y_pred = predict(model, X_test_norm)
        y_true = y[test_idx]

        acc = accuracy_score(y_true, y_pred)
        macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

        print(
            f"\nERGEBNIS Fold {fold_no} | Testperson {test_subject} | "
            f"Accuracy: {acc:.4f} | Macro-F1: {macro_f1:.4f}"
        )

        results.append({
            "fold": fold_no,
            "test_subject": test_subject,
            "validation_subject": val_subject,
            "n_train_windows": len(fit_idx),
            "n_val_windows": len(val_idx),
            "n_test_windows": len(test_idx),
            "best_val_loss": best_val_loss,
            "accuracy": acc,
            "macro_f1": macro_f1,
        })

        all_true.append(y_true)
        all_pred.append(y_pred)

        # Speicher freigeben, besonders bei GPU-Training.
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    y_true_all = np.concatenate(all_true)
    y_pred_all = np.concatenate(all_pred)

    results_df = pd.DataFrame(results)
    results_df.to_csv(OUTPUT_CSV, index=False)

    print("\n" + "=" * 78)
    print("GESAMTERGEBNIS – Leave-One-Subject-Out")
    print(f"Accuracy pro Fold: {results_df['accuracy'].mean():.4f} "
          f"± {results_df['accuracy'].std(ddof=1):.4f}")
    print(f"Macro-F1 pro Fold: {results_df['macro_f1'].mean():.4f} "
          f"± {results_df['macro_f1'].std(ddof=1):.4f}")
    print(f"Gesamte gepoolte Accuracy: {accuracy_score(y_true_all, y_pred_all):.4f}")
    print(f"Gesamter gepoolter Macro-F1: "
          f"{f1_score(y_true_all, y_pred_all, average='macro', zero_division=0):.4f}")

    print("\nKlassifikationsbericht (alle zurückgehaltenen Testpersonen zusammen):")
    print(classification_report(
        y_true_all,
        y_pred_all,
        labels=np.arange(n_classes),
        target_names=[str(c) for c in label_encoder.classes_],
        zero_division=0,
    ))

    print(f"Fold-Ergebnisse gespeichert unter: {OUTPUT_CSV.resolve()}")


if __name__ == "__main__":
    main()


PyTorch-Version: 2.14.0+rocm10.1.0
Gerät: cuda
GPU: AMD Radeon RX 6600 XT

Gefundene Dateien: 18
[1/18] S10_A1_E1.mat: 889 Fenster, 10 EMG-Kanäle, Person 10
[2/18] S16_A1_E1.mat: 719 Fenster, 10 EMG-Kanäle, Person 16
[3/18] S17_A1_E1.mat: 718 Fenster, 10 EMG-Kanäle, Person 17
[4/18] S18_A1_E1.mat: 859 Fenster, 10 EMG-Kanäle, Person 18
[5/18] S19_A1_E1.mat: 826 Fenster, 10 EMG-Kanäle, Person 19
[6/18] S1_A1_E1.mat: 707 Fenster, 10 EMG-Kanäle, Person 1
[7/18] S23_A1_E1.mat: 886 Fenster, 10 EMG-Kanäle, Person 23
[8/18] S24_A1_E1.mat: 840 Fenster, 10 EMG-Kanäle, Person 24
[9/18] S25_A1_E1.mat: 942 Fenster, 10 EMG-Kanäle, Person 25
[10/18] S26_A1_E1.mat: 852 Fenster, 10 EMG-Kanäle, Person 26
[11/18] S27_A1_E1.mat: 1092 Fenster, 10 EMG-Kanäle, Person 27
[12/18] S2_A1_E1.mat: 833 Fenster, 10 EMG-Kanäle, Person 2
[13/18] S3_A1_E1.mat: 739 Fenster, 10 EMG-Kanäle, Person 3
[14/18] S4_A1_E1.mat: 925 Fenster, 10 EMG-Kanäle, Person 4
[15/18] S5_A1_E1.mat: 1094 Fenster, 10 EMG-Kanäle, Person 5
[16/1

MIOpen: Warning [OpenRuntimeLibraryForDevice] CK grouped conv library not found for device gfx1032: libMIOpenCKGroupedConv_gfx1032.so: cannot open shared object file: No such file or directory


  Training: Batch 20/219
  Training: Batch 40/219
  Training: Batch 60/219
  Training: Batch 80/219
  Training: Batch 100/219
  Training: Batch 120/219
  Training: Batch 140/219
  Training: Batch 160/219
  Training: Batch 180/219
  Training: Batch 200/219
  Training: Batch 219/219
  Validierung: Batch 12/12
Fold 01 | Epoch 01/60 | Train Loss 1.9391 Acc 0.3261 | Val Loss 2.2429 Acc 0.2490 | LR 1.00e-03
  Training: Batch 20/219
  Training: Batch 40/219
  Training: Batch 60/219
  Training: Batch 80/219
  Training: Batch 100/219
  Training: Batch 120/219
  Training: Batch 140/219
  Training: Batch 160/219
  Training: Batch 180/219
  Training: Batch 200/219
  Training: Batch 219/219
  Validierung: Batch 12/12
Fold 01 | Epoch 02/60 | Train Loss 1.4029 Acc 0.5041 | Val Loss 2.8324 Acc 0.2935 | LR 1.00e-03
  Training: Batch 20/219
  Training: Batch 40/219
  Training: Batch 60/219
  Training: Batch 80/219
  Training: Batch 100/219
  Training: Batch 120/219
  Training: Batch 140/219
  Training: 

In [2]:
from pathlib import Path
import re, warnings
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.signal import butter, sosfiltfilt, welch
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Diese Einstellungen mit dem CNN-Skript abgleichen
DATA_DIR = Path("./Data")
FILE_PATTERN = "*_E1.mat"
FS, LOWCUT, HIGHCUT = 2000, 20, 450
APPLY_BANDPASS = True
WINDOW_SIZE, STEP_SIZE = 400, 50
N_ESTIMATORS, RANDOM_SEED, N_JOBS = 300, 42, -1
OUTPUT_CSV = Path("random_forest_loso_results.csv")
OUTPUT_REPORT = Path("random_forest_loso_report.txt")
OUTPUT_CONFUSION = Path("random_forest_loso_confusion_matrix.csv")


def get_subject_id(path):
    match = re.search(r"S(\d+)", path.name, re.IGNORECASE)
    if not match:
        raise ValueError(f"Personen-ID nicht im Dateinamen gefunden: {path.name}")
    return int(match.group(1))


def load_emg_and_labels(path):
    data = sio.loadmat(path)
    if "emg" not in data:
        raise KeyError(f"'emg' fehlt in {path}")
    emg = np.asarray(data["emg"], dtype=np.float32)
    if emg.ndim != 2:
        raise ValueError(f"Unerwartete EMG-Form {emg.shape} in {path}")
    # Identisch zur CNN-Version: restimulus bevorzugen, sonst stimulus.
    label_key = "restimulus" if "restimulus" in data else "stimulus"
    if label_key not in data:
        raise KeyError(f"Weder 'restimulus' noch 'stimulus' in {path}")
    labels = np.rint(np.asarray(data[label_key]).reshape(-1)).astype(np.int64)
    n = min(len(emg), len(labels))
    emg, labels = emg[:n], labels[:n]
    if APPLY_BANDPASS:
        if not (0 < LOWCUT < HIGHCUT < FS / 2):
            raise ValueError("Ungültige Bandpass-Grenzen")
        sos = butter(4, [LOWCUT, HIGHCUT], btype="bandpass", fs=FS, output="sos")
        if len(emg) > 30:
            emg = sosfiltfilt(sos, emg, axis=0).astype(np.float32)
    return emg, labels


def make_windows(emg, labels):
    windows, window_labels = [], []
    if len(emg) < WINDOW_SIZE:
        return np.empty((0, WINDOW_SIZE, emg.shape[1]), np.float32), np.empty(0, np.int64)
    for start in range(0, len(emg) - WINDOW_SIZE + 1, STEP_SIZE):
        end = start + WINDOW_SIZE
        values, counts = np.unique(labels[start:end], return_counts=True)
        label = int(values[np.argmax(counts)])
        if label == 0:
            continue
        windows.append(emg[start:end])
        window_labels.append(label)
    if not windows:
        return np.empty((0, WINDOW_SIZE, emg.shape[1]), np.float32), np.empty(0, np.int64)
    return np.asarray(windows, np.float32), np.asarray(window_labels, np.int64)


def load_dataset():
    files = sorted(DATA_DIR.rglob(FILE_PATTERN))
    if not files:
        raise FileNotFoundError(f"Keine {FILE_PATTERN}-Dateien unter {DATA_DIR.resolve()} gefunden")
    xs, ys, groups = [], [], []
    print(f"Gefundene Dateien: {len(files)}")
    for i, path in enumerate(files, 1):
        try:
            subject = get_subject_id(path)
            emg, labels = load_emg_and_labels(path)
            x_file, y_file = make_windows(emg, labels)
            if not len(y_file):
                print(f"[{i}/{len(files)}] Übersprungen (keine aktiven Fenster): {path}")
                continue
            xs.append(x_file); ys.append(y_file)
            groups.append(np.full(len(y_file), subject, dtype=np.int64))
            print(f"[{i}/{len(files)}] {path.name}: {len(y_file)} Fenster, {emg.shape[1]} Kanäle, Person {subject}")
        except Exception as exc:
            warnings.warn(f"Datei übersprungen: {path} ({exc})")
    if not xs:
        raise RuntimeError("Keine gültigen EMG-Fenster geladen")
    X = np.concatenate(xs).astype(np.float32)
    y = np.concatenate(ys).astype(np.int64)
    groups = np.concatenate(groups).astype(np.int64)
    print("\nDatensatz:", X.shape, "| Personen:", np.unique(groups).tolist(), "| Klassen:", np.unique(y).tolist())
    return X, y, groups


def extract_features(window):
    "Acht EMG-Features pro Kanal: RMS, MAV, WL, VAR, ZC, SSC, MDF, Entropie."
    features = []
    for ch in range(window.shape[1]):
        sig = np.asarray(window[:, ch], dtype=np.float64)
        rms = np.sqrt(np.mean(sig ** 2))
        mav = np.mean(np.abs(sig))
        wl = np.sum(np.abs(np.diff(sig)))
        var = np.var(sig)
        zc = np.sum(sig[:-1] * sig[1:] < 0)
        diff = np.diff(sig)
        ssc = np.sum(diff[:-1] * diff[1:] < 0)
        freqs, psd = welch(sig, fs=FS, nperseg=min(256, len(sig)))
        power = np.sum(psd)
        if power > 0 and np.isfinite(power):
            median_idx = min(np.searchsorted(np.cumsum(psd), power / 2), len(freqs) - 1)
            median_freq = freqs[median_idx]
            probs = psd[psd > 0] / power
            entropy = -np.sum(probs * np.log2(probs))
            if len(psd) > 1:
                entropy /= np.log2(len(psd))
        else:
            median_freq, entropy = 0.0, 0.0
        features.extend([rms, mav, wl, var, zc, ssc, median_freq, entropy])
    return np.asarray(features, dtype=np.float32)


def extract_all_features(X):
    out = []
    print(f"\nBerechne Features für {len(X)} Fenster ...")
    for i, window in enumerate(X, 1):
        out.append(extract_features(window))
        if i % 1000 == 0 or i == len(X):
            print(f"  Features: {i}/{len(X)}", flush=True)
    return np.asarray(out, dtype=np.float32)


def main():
    X_windows, y_raw, groups = load_dataset()
    X = extract_all_features(X_windows)
    encoder = LabelEncoder()
    y = encoder.fit_transform(y_raw)
    n_classes = len(encoder.classes_)
    subjects = np.unique(groups)
    logo = LeaveOneGroupOut()
    results, all_true, all_pred = [], [], []
    print(f"\nStarte Random-Forest-LOSO: {len(subjects)} Personen, {n_classes} Klassen")
    print("Die Validierungsperson wird nach derselben Regel wie im CNN aus dem Fit ausgeschlossen.")

    for fold_no, (trainval_idx, test_idx) in enumerate(logo.split(X, y, groups), 1):
        test_subject = int(groups[test_idx][0])
        trainval_subjects = np.unique(groups[trainval_idx])
        if len(trainval_subjects) < 2:
            raise RuntimeError("Mindestens drei Personen insgesamt erforderlich")
        # Gleiche deterministische Validierungsperson-Regel wie im CNN-Skript.
        rng = np.random.default_rng(42 + fold_no)
        val_subject = int(rng.choice(trainval_subjects))
        fit_idx = trainval_idx[~np.isin(groups[trainval_idx], [val_subject])]
        print("\n" + "=" * 68)
        print(f"Fold {fold_no}/{len(subjects)} | Testperson: {test_subject} | ausgelassene Validierungsperson: {val_subject}")
        print(f"Trainingspersonen: {np.unique(groups[fit_idx]).tolist()}")
        print(f"Train-Fenster: {len(fit_idx)} | Test-Fenster: {len(test_idx)}")

        model = RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_SEED,
                                       n_jobs=N_JOBS, class_weight="balanced_subsample")
        model.fit(X[fit_idx], y[fit_idx])
        pred = model.predict(X[test_idx])
        true = y[test_idx]
        acc = accuracy_score(true, pred)
        mf1 = f1_score(true, pred, labels=np.arange(n_classes), average="macro", zero_division=0)
        print(f"Accuracy: {acc:.4f} | Macro-F1: {mf1:.4f}")
        results.append({"fold": fold_no, "test_subject": test_subject, "validation_subject": val_subject,
                        "n_train_windows": len(fit_idx), "n_test_windows": len(test_idx),
                        "accuracy": acc, "macro_f1": mf1})
        all_true.append(true); all_pred.append(pred)
        # Nach jedem Fold sichern, damit bei einem Abbruch Ergebnisse erhalten bleiben.
        pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)

    true_all, pred_all = np.concatenate(all_true), np.concatenate(all_pred)
    df = pd.DataFrame(results)
    df.to_csv(OUTPUT_CSV, index=False)
    cm = confusion_matrix(true_all, pred_all, labels=np.arange(n_classes))
    pd.DataFrame(cm, index=[f"true_{c}" for c in encoder.classes_],
                 columns=[f"pred_{c}" for c in encoder.classes_]).to_csv(OUTPUT_CONFUSION)
    report = classification_report(true_all, pred_all, labels=np.arange(n_classes),
                                   target_names=[str(c) for c in encoder.classes_], zero_division=0)
    summary = ("GESAMTERGEBNIS – Random Forest LOSO\n"
               f"Accuracy pro Fold: {df.accuracy.mean():.4f} ± {df.accuracy.std(ddof=1):.4f}\n"
               f"Macro-F1 pro Fold: {df.macro_f1.mean():.4f} ± {df.macro_f1.std(ddof=1):.4f}\n"
               f"Gepoolte Accuracy: {accuracy_score(true_all, pred_all):.4f}\n"
               f"Gepoolter Macro-F1: {f1_score(true_all, pred_all, labels=np.arange(n_classes), average='macro', zero_division=0):.4f}\n\n"
               f"Klassifikationsbericht:\n{report}\n\nFold-Ergebnisse: {OUTPUT_CSV.resolve()}\n"
               f"Konfusionsmatrix: {OUTPUT_CONFUSION.resolve()}")
    print("\n" + "=" * 68 + "\n" + summary)
    OUTPUT_REPORT.write_text(summary, encoding="utf-8")
    print(f"Bericht gespeichert unter: {OUTPUT_REPORT.resolve()}")

if __name__ == "__main__":
    main()


Gefundene Dateien: 18
[1/18] S10_A1_E1.mat: 889 Fenster, 10 Kanäle, Person 10
[2/18] S16_A1_E1.mat: 719 Fenster, 10 Kanäle, Person 16
[3/18] S17_A1_E1.mat: 718 Fenster, 10 Kanäle, Person 17
[4/18] S18_A1_E1.mat: 859 Fenster, 10 Kanäle, Person 18
[5/18] S19_A1_E1.mat: 826 Fenster, 10 Kanäle, Person 19
[6/18] S1_A1_E1.mat: 707 Fenster, 10 Kanäle, Person 1
[7/18] S23_A1_E1.mat: 886 Fenster, 10 Kanäle, Person 23
[8/18] S24_A1_E1.mat: 840 Fenster, 10 Kanäle, Person 24
[9/18] S25_A1_E1.mat: 942 Fenster, 10 Kanäle, Person 25
[10/18] S26_A1_E1.mat: 852 Fenster, 10 Kanäle, Person 26
[11/18] S27_A1_E1.mat: 1092 Fenster, 10 Kanäle, Person 27
[12/18] S2_A1_E1.mat: 833 Fenster, 10 Kanäle, Person 2
[13/18] S3_A1_E1.mat: 739 Fenster, 10 Kanäle, Person 3
[14/18] S4_A1_E1.mat: 925 Fenster, 10 Kanäle, Person 4
[15/18] S5_A1_E1.mat: 1094 Fenster, 10 Kanäle, Person 5
[16/18] S6_A1_E1.mat: 950 Fenster, 10 Kanäle, Person 6
[17/18] S7_A1_E1.mat: 845 Fenster, 10 Kanäle, Person 7
[18/18] S8_A1_E1.mat: 721 Fens